# MVA Hackathon 2026 — Track 1 · full pipeline (yasld)
**Phenotype-guided, selection-corrected variant prioritisation for one proband.** Self-contained: the first cells write the
`mvapipe` package to disk, then each stage runs in order (stages 8–11 are supplementary analyses).

**Antes de ejecutar:** (1) acepta las condiciones del dataset `SageBio/mva-hackathon-2026-data` en Hugging Face;
(2) Kaggle → *Add-ons → Secrets*: `HF_TOKEN` (token de lectura) y **Internet ON**; (3) *Run all*. Las etapas 2 y 4 tardan varios
minutos; cada etapa guarda su salida en disco, así que se puede reanudar. Si una etapa falla, relanza solo esa celda.

**Datos protegidos.** Es un niño real: no publiques el VCF, los parquet, el JSON de anotación ni la tabla del locus.

## 0. Setup

In [ ]:
!pip install -q cyvcf2 python-docx huggingface_hub pyarrow
import os, sys
os.makedirs("mvapipe", exist_ok=True)
sys.path.insert(0, os.getcwd())

In [ ]:
%%writefile mvapipe/__init__.py
"""MVA Hackathon 2026 - Track 1 pipeline (single-proband, phenotype-guided variant prioritisation)."""
__version__ = "1.0.0"

In [ ]:
%%writefile mvapipe/config.py
"""Central configuration.

Every threshold and constant used in the report lives here, so that the methods
text and the code cannot drift apart.
"""
import os
from pathlib import Path

REPO_ID = "SageBio/mva-hackathon-2026-data"          # gated dataset (accept the terms first)
VCF_NAME = "WGS_EX2312012_HGWCNDSX7.vcf.gz"
TBI_NAME = VCF_NAME + ".tbi"
PHENO_NAME = "Challenge_Clinical_Phenotype_1.docx"

GTF_URL = os.environ.get(
    "MVA_GTF_URL",
    "https://ftp.ensembl.org/pub/release-112/gtf/homo_sapiens/Homo_sapiens.GRCh38.112.chr.gtf.gz",
)
HPO_URL = os.environ.get(
    "MVA_HPO_URL",
    "https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/phenotype_to_genes.txt",
)

PROBAND_ID = "PROBAND01"

# The patient's HPO terms are NOT stored in the code. Stage 1 reads them from the challenge phenotype
# document and writes them to the (git-ignored) working directory; stage 6 loads them from there.
HPO_TERMS: dict = {}


def load_hpo_terms() -> dict:
    """Return the proband's HPO terms {id: label}: in memory if set, else from the working directory."""
    if HPO_TERMS:
        return HPO_TERMS
    import json as _json
    f = workdir() / FILES["hpo_terms"]
    if not f.exists():
        raise FileNotFoundError("hpo_terms.json not found: run stage 1 first (it reads the phenotype document)")
    HPO_TERMS.update(_json.loads(f.read_text()))
    return HPO_TERMS

QC_MIN_DP = 10
QC_MIN_GQ = 30
CDS_PAD = 2
RARE_AF = 0.001

IMPACT_HIGH = {"stop_gained", "frameshift_variant", "splice_acceptor_variant",
               "splice_donor_variant", "start_lost", "stop_lost"}
IMPACT_MODERATE = {"missense_variant", "inframe_insertion", "inframe_deletion",
                   "protein_altering_variant"}
TRUNCATING = {"stop_gained", "frameshift_variant", "splice_acceptor_variant", "splice_donor_variant"}

# Damage weights (set after the candidate was known: NOT blind, see report)
W_STRONG = 1.0
W_OTHER = 0.5
W_SINGLE_ALLELE = 0.25
W_HOM_MULT = 2.0

PERM_N = 2000
PERM_SEED = 42
IC_TOL = 0.5

MYVARIANT_BATCH = 500
MYVARIANT_FIELDS = ",".join([
    "clinvar.rcv.clinical_significance",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "snpeff.ann.effect", "snpeff.ann.hgvs_p", "snpeff.ann.genename",
    "dbnsfp.cadd.phred", "dbnsfp.revel", "dbnsfp.genename",
])

EPCR_PRIMARY = 0.60
EPCR_SECONDARY = 0.05
MAX_ROWS = 10

LOCUS_FLANK = 50_000
BIN_SIZE = 5_000_000
BAF_MIN_DP = 20
BAF_MIN_SITES_PER_BIN = 200


def workdir() -> Path:
    """Working directory: $MVA_WORKDIR, else /kaggle/working, else ./work."""
    env = os.environ.get("MVA_WORKDIR")
    if env:
        p = Path(env)
    elif Path("/kaggle/working").exists():
        p = Path("/kaggle/working")
    else:
        p = Path("./work")
    p.mkdir(parents=True, exist_ok=True)
    return p


FILES = {
    "vcf": "data/" + VCF_NAME, "tbi": "data/" + TBI_NAME, "pheno": "data/" + PHENO_NAME,
    "gtf": "genes.gtf.gz", "hpo": "phenotype_to_genes.txt",
    "qc": "variants_pass.parquet", "qc_stats": "qc_stats.json",
    "coding": "variants_coding.parquet", "myvariant_raw": "myvariant_raw.json",
    "annotated": "annotated.parquet",
    "scan_A": "scan_A_homozygous.csv", "scan_B": "scan_B_compound_het.csv",
    "scan_clinvar": "scan_clinvar_pathogenic.csv", "scan_summary": "scan_summary.json",
    "ranking": "phenotype_ranking.csv", "loo": "phenotype_leave_one_out.csv",
    "perm": "permutation_results.json", "selected": "selected_candidates.json",
    "verification": "verification.json",
    "locus": "locus_review.csv", "locus_summary": "locus_summary.json",
    "nmd": "nmd_check.json", "gnomad": "gnomad_check.json",
    "baf_chrom": "baf_by_chromosome.csv", "baf_bins": "baf_bins.csv", "baf_summary": "baf_summary.json",
    "hpo_terms": "hpo_terms.json", "submission": "submission.csv", "provenance": "provenance.json",
}


def path(key: str) -> Path:
    p = workdir() / FILES[key]
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

In [ ]:
%%writefile mvapipe/utils.py
"""Shared helpers."""
from __future__ import annotations

import gzip
import hashlib
import json
import os
import re
from typing import Iterator

import numpy as np

ATTR_RE = re.compile(r'(\w+) "([^"]*)"')
GENE_RE = re.compile(r'gene_name "([^"]+)"')


def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def flat_nums(x) -> list[float]:
    """Flatten any mixture of numbers / lists / dicts with a 'score' key into floats."""
    out: list[float] = []
    for v in as_list(x):
        if isinstance(v, dict):
            out += flat_nums(v.get("score"))
        elif isinstance(v, list):
            out += flat_nums(v)
        elif isinstance(v, (int, float)) and not isinstance(v, bool):
            out.append(float(v))
    return out


def maxnum(x) -> float:
    n = flat_nums(x)
    return max(n) if n else float("nan")


def hgvs_genomic(chrom: str, pos: int, ref: str, alt: str) -> str | None:
    """Genomic HGVS string (hg38) for MyVariant.info. Multi-allelic records return None."""
    chrom = str(chrom).replace("chr", "")
    if "," in alt:
        return None
    if len(ref) == len(alt) == 1:
        return f"chr{chrom}:g.{pos}{ref}>{alt}"
    if len(ref) > len(alt) and ref.startswith(alt):
        s, e = pos + len(alt), pos + len(ref) - 1
        return f"chr{chrom}:g.{s}del" if s == e else f"chr{chrom}:g.{s}_{e}del"
    if len(alt) > len(ref) and alt.startswith(ref):
        q = pos + len(ref) - 1
        return f"chr{chrom}:g.{q}_{q + 1}ins{alt[len(ref):]}"
    if len(ref) == 1:
        return f"chr{chrom}:g.{pos}delins{alt}"
    return f"chr{chrom}:g.{pos}_{pos + len(ref) - 1}delins{alt}"


def merge_intervals(intervals: list[tuple[int, int]]) -> tuple[np.ndarray, np.ndarray]:
    if not intervals:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64)
    intervals = sorted(intervals)
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    a = np.array(merged, dtype=np.int64)
    return a[:, 0], a[:, 1]


def in_intervals(starts: np.ndarray, ends: np.ndarray, pos: np.ndarray) -> np.ndarray:
    if len(starts) == 0:
        return np.zeros(len(pos), dtype=bool)
    i = np.searchsorted(starts, pos, side="right") - 1
    return (i >= 0) & (pos <= ends[np.clip(i, 0, None)])


def gtf_iter(gtf_path, features: set[str] | None = None) -> Iterator[tuple]:
    opener = gzip.open if str(gtf_path).endswith(".gz") else open
    with opener(gtf_path, "rt") as f:
        for line in f:
            if line[0] == "#":
                continue
            c = line.rstrip("\n").split("\t")
            if len(c) < 9:
                continue
            if features is not None and c[2] not in features:
                continue
            yield c[0], c[2], int(c[3]), int(c[4]), c[6], c[8]


def parse_attrs(attr_string: str) -> dict:
    d: dict = {}
    for k, v in ATTR_RE.findall(attr_string):
        if k in d:
            if not isinstance(d[k], list):
                d[k] = [d[k]]
            d[k].append(v)
        else:
            d[k] = v
    return d


def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_json(obj, path) -> None:
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def read_json(path):
    with open(path) as f:
        return json.load(f)


def hf_token() -> str | None:
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def fmt(v, tag: str):
    """Read a FORMAT field from a cyvcf2 Variant; None when the tag is not declared in the header
    (cyvcf2 raises KeyError for undeclared tags, e.g. PGT/PID in an unphased callset)."""
    try:
        return v.format(tag)
    except KeyError:
        return None

In [ ]:
%%writefile mvapipe/stage01_03_data_qc_coding.py
"""Stages 1-3: data download (with provenance), VCF quality control, coding-region restriction."""
from __future__ import annotations

import datetime as dt
import re
from collections import Counter

import numpy as np
import pandas as pd
import requests

from . import config as cfg
from .utils import (fmt, gtf_iter, hf_token, in_intervals, merge_intervals,
                    sha256_file, write_json)


def _download(url: str, dest) -> None:
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)


def extract_hpo_terms(docx_path) -> dict[str, str]:
    """Read HPO identifiers and labels from the challenge phenotype document."""
    from docx import Document
    terms: dict[str, str] = {}
    doc = Document(docx_path)
    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells]
            for i, c in enumerate(cells):
                if re.fullmatch(r"HP:\d{7}", c):
                    terms[c] = cells[i - 1] if i > 0 else ""
    return terms


def stage01_data(force: bool = False) -> dict:
    from huggingface_hub import hf_hub_download

    token = hf_token()
    wd = cfg.workdir()
    for name in (cfg.VCF_NAME, cfg.TBI_NAME, cfg.PHENO_NAME):
        dest = wd / "data" / name
        if dest.exists() and not force:
            continue
        hf_hub_download(cfg.REPO_ID, filename=name, repo_type="dataset",
                        local_dir=str(wd / "data"), token=token)
    for key, url in (("gtf", cfg.GTF_URL), ("hpo", cfg.HPO_URL)):
        dest = cfg.path(key)
        if not dest.exists() or force:
            _download(url, dest)

    prov = {"date_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "gtf_url": cfg.GTF_URL, "hpo_url": cfg.HPO_URL, "files": {}}
    for key in ("tbi", "pheno", "gtf", "hpo"):
        p = cfg.path(key)
        prov["files"][p.name] = {"bytes": p.stat().st_size, "sha256": sha256_file(p)}
    vcf = cfg.path("vcf")
    prov["files"][vcf.name] = {"bytes": vcf.stat().st_size}
    try:
        found = extract_hpo_terms(cfg.path("pheno"))
        write_json(found, cfg.path("hpo_terms"))          # private: stays in the working directory
        cfg.HPO_TERMS.clear(); cfg.HPO_TERMS.update(found)
        prov["n_hpo_terms_in_document"] = len(found)
    except Exception as e:
        prov["hpo_terms_in_document"] = f"could not parse: {e}"
    write_json(prov, cfg.path("provenance"))
    return prov


def stage02_qc(vcf_path=None, out=None, limit: int | None = None) -> dict:
    """Keep PASS, non-reference, DP >= 10, GQ >= 30 calls; record every exclusion count."""
    from cyvcf2 import VCF

    vcf_path = vcf_path or cfg.path("vcf")
    out = out or cfg.path("qc")
    counts: Counter = Counter()
    rows = []
    for v in VCF(str(vcf_path)):
        counts["total_records"] += 1
        if v.FILTER is not None:
            counts["excluded_filter_not_pass"] += 1
        else:
            gt = int(v.gt_types[0])
            if gt in (0, 2):
                counts["excluded_reference_or_unknown_gt"] += 1
            else:
                dp, gq = int(v.gt_depths[0]), int(v.gt_quals[0])
                if dp < cfg.QC_MIN_DP or gq < cfg.QC_MIN_GQ:
                    counts["excluded_low_dp_or_gq"] += 1
                else:
                    ad = fmt(v, "AD")
                    ad_ref, ad_alt = (int(ad[0][0]), int(ad[0][1])) if ad is not None else (-1, -1)
                    rows.append((v.CHROM, v.POS, v.REF, ",".join(v.ALT), v.QUAL, gt, dp, gq,
                                 bool(v.INFO.get("DB", False)), ad_ref, ad_alt))
                    counts["kept"] += 1
        if limit and counts["total_records"] >= limit:
            break

    df = pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "qual", "gt", "dp", "gq",
                                     "in_dbsnp", "ad_ref", "ad_alt"])
    df["chrom"] = df["chrom"].astype(str)
    df.to_parquet(out)
    stats = dict(counts)
    stats["het"] = int((df["gt"] == 1).sum())
    stats["hom_alt"] = int((df["gt"] == 3).sum())
    stats["in_dbsnp"] = int(df["in_dbsnp"].sum())
    stats["not_in_dbsnp"] = int((~df["in_dbsnp"]).sum())
    write_json(stats, cfg.path("qc_stats"))
    return stats


def cds_intervals(gtf_path, pad: int = cfg.CDS_PAD) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    raw: dict[str, list[tuple[int, int]]] = {}
    for chrom, _feat, s, e, _strand, _attrs in gtf_iter(gtf_path, {"CDS"}):
        raw.setdefault(chrom, []).append((s - pad, e + pad))
    return {ch: merge_intervals(iv) for ch, iv in raw.items()}


def stage03_coding(qc_path=None, gtf_path=None, out=None) -> dict:
    qc_path = qc_path or cfg.path("qc")
    gtf_path = gtf_path or cfg.path("gtf")
    out = out or cfg.path("coding")
    df = pd.read_parquet(qc_path)
    merged = cds_intervals(gtf_path)

    mask = np.zeros(len(df), dtype=bool)
    pos_all = df["pos"].to_numpy()
    for chrom, idx in df.groupby("chrom").indices.items():
        if chrom in merged:
            s, e = merged[chrom]
            mask[idx] = in_intervals(s, e, pos_all[idx])

    coding = df[mask].reset_index(drop=True)
    coding.to_parquet(out)
    stats = {"input_variants": int(len(df)), "coding_variants": int(len(coding)),
             "coding_het": int((coding["gt"] == 1).sum()),
             "coding_hom_alt": int((coding["gt"] == 3).sum())}
    write_json(stats, cfg.workdir() / "coding_stats.json")
    return stats

In [ ]:
%%writefile mvapipe/stage04_05_annotate_scan.py
"""Stages 4-5: batched annotation (MyVariant.info) and the pre-specified, phenotype-agnostic scan."""
from __future__ import annotations

import json
import time

import numpy as np
import pandas as pd
import requests

from . import config as cfg
from .utils import GENE_RE, as_list, flat_nums, gtf_iter, hgvs_genomic, maxnum, write_json


def add_hgvs(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hgvs"] = [hgvs_genomic(c, p, r, a) for c, p, r, a in
                  zip(df["chrom"], df["pos"], df["ref"], df["alt"])]
    return df


def stage04_annotate(coding_path=None, force: bool = False, session=None) -> dict:
    """POST variants to MyVariant.info in batches; the raw JSON is cached on disk."""
    coding_path = coding_path or cfg.path("coding")
    raw_path = cfg.path("myvariant_raw")
    df = add_hgvs(pd.read_parquet(coding_path))
    df.to_parquet(cfg.workdir() / "variants_coding_hgvs.parquet")
    n_multi = int(df["hgvs"].isna().sum())

    if raw_path.exists() and not force:
        with open(raw_path) as fh:
            raw = json.load(fh)
    else:
        http = session or requests
        ids = df["hgvs"].dropna().unique().tolist()
        raw = []
        for i in range(0, len(ids), cfg.MYVARIANT_BATCH):
            batch = ids[i:i + cfg.MYVARIANT_BATCH]
            for attempt in range(3):
                r = http.post("https://myvariant.info/v1/variant",
                              data={"ids": ",".join(batch), "assembly": "hg38",
                                    "fields": cfg.MYVARIANT_FIELDS}, timeout=120)
                if r.status_code == 200:
                    raw.extend(r.json())
                    break
                time.sleep(3 * (attempt + 1))
            else:
                raise RuntimeError(f"MyVariant batch starting at {i} failed after 3 attempts")
        with open(raw_path, "w") as fh:
            json.dump(raw, fh)
    stats = {"coding_variants": int(len(df)), "multiallelic_not_annotated": n_multi,
             "responses": len(raw), "notfound": sum(1 for x in raw if x.get("notfound"))}
    write_json(stats, cfg.workdir() / "annotation_stats.json")
    return stats


def parse_record(x: dict) -> dict:
    """One MyVariant record -> flat dict (worst effect over transcripts; max AF; ClinVar flags)."""
    out = {"hgvs": x.get("query")}
    best = None
    for a in as_list((x.get("snpeff") or {}).get("ann")):
        eff = str(a.get("effect", ""))
        parts = set(eff.split("&"))
        rank = 2 if parts & cfg.IMPACT_HIGH else 1 if parts & cfg.IMPACT_MODERATE else 0
        if best is None or rank > best[0]:
            best = (rank, eff, a.get("genename") or a.get("gene_name"), a.get("hgvs_p"))
    out["impacto"], out["efecto"], out["gen_api"], out["hgvs_p"] = best if best else (np.nan,) * 4

    afs: list[float] = []
    for k in ("gnomad_genome", "gnomad_exome"):
        afs += flat_nums(((x.get(k) or {}).get("af") or {}).get("af"))
    out["gnomad_af"] = max(afs) if afs else np.nan

    cs = [str(r.get("clinical_significance", ""))
          for r in as_list((x.get("clinvar") or {}).get("rcv"))]
    out["clinvar_patog"] = any(("athogenic" in c) and ("Conflicting" not in c) for c in cs)
    out["clinvar_benigno"] = any("enign" in c for c in cs)
    dbn = x.get("dbnsfp") or {}
    cadd = dbn.get("cadd")
    out["cadd"] = maxnum(cadd.get("phred")) if isinstance(cadd, dict) else float("nan")
    out["revel"] = maxnum(dbn.get("revel"))
    return out


def assign_genes_local(t: pd.DataFrame, gtf_path) -> pd.Series:
    """Assign a gene symbol from CDS intervals in the GTF (no web service needed)."""
    rows = []
    for chrom, _f, s, e, _st, attrs in gtf_iter(gtf_path, {"CDS"}):
        m = GENE_RE.search(attrs)
        if m:
            rows.append((chrom, s - cfg.CDS_PAD, e + cfg.CDS_PAD, m.group(1)))
    cds = pd.DataFrame(rows, columns=["chrom", "s", "e", "gene"]).drop_duplicates()

    gene = np.full(len(t), None, dtype=object)
    assigned = np.zeros(len(t), dtype=bool)
    chrom_arr = t["chrom"].to_numpy()
    pos_arr = t["pos"].to_numpy()
    for ch, sub in cds.groupby("chrom"):
        sub = sub.sort_values("s")
        s, e, g = sub["s"].to_numpy(), sub["e"].to_numpy(), sub["gene"].to_numpy()
        ix = np.where(chrom_arr == ch)[0]
        if len(ix) == 0:
            continue
        pos = pos_arr[ix]
        k = np.searchsorted(s, pos, side="right") - 1
        for back in range(4):
            kk = k - back
            ok = (kk >= 0) & (pos <= e[np.clip(kk, 0, None)]) & ~assigned[ix]
            for j in np.where(ok)[0]:
                gene[ix[j]] = g[kk[j]]
                assigned[ix[j]] = True
    return pd.Series(gene, index=t.index)


def stage05_scan(hgvs_path=None, raw_path=None, gtf_path=None) -> dict:
    hgvs_path = hgvs_path or cfg.workdir() / "variants_coding_hgvs.parquet"
    raw_path = raw_path or cfg.path("myvariant_raw")
    gtf_path = gtf_path or cfg.path("gtf")

    exo = pd.read_parquet(hgvs_path)
    with open(raw_path) as fh:
        raw = json.load(fh)
    ann = pd.DataFrame([parse_record(x) for x in raw if not x.get("notfound")]) \
        .drop_duplicates("hgvs")
    t = exo.merge(ann, on="hgvs", how="left")

    t["gen_local"] = assign_genes_local(t, gtf_path)
    t["gen"] = t["gen_local"].fillna(t["gen_api"])

    t["rara"] = t["gnomad_af"].lt(cfg.RARE_AF)
    t["sin_dato_gnomad"] = t["gnomad_af"].isna()
    t["rara_o_sin_dato"] = t["rara"] | t["sin_dato_gnomad"]
    t["alto_mod"] = t["impacto"].isin([1, 2])
    for c in ("clinvar_patog", "clinvar_benigno", "rara", "alto_mod", "rara_o_sin_dato"):
        t[c] = t[c].astype("boolean").fillna(False).astype(bool)
    t.to_parquet(cfg.path("annotated"))

    cols = ["gen", "chrom", "pos", "ref", "alt", "gt", "efecto", "hgvs_p", "gnomad_af",
            "clinvar_patog", "cadd", "revel", "dp", "gq", "ad_ref", "ad_alt"]
    base = t[t["alto_mod"] & t["rara_o_sin_dato"] & ~t["clinvar_benigno"]]

    A = base[base["gt"] == 3].sort_values("gen")
    het = base[base["gt"] == 1]
    n = het.groupby("gen").size()
    B = het[het["gen"].isin(n[n >= 2].index)].sort_values(["gen", "pos"])
    efecto_grave = t["efecto"].fillna("").apply(lambda e: bool(set(e.split("&")) & cfg.TRUNCATING))
    genes_trunc = sorted(set(B[efecto_grave.reindex(B.index)]["gen"]))
    cp = t[t["clinvar_patog"] & ~t["clinvar_benigno"]].sort_values("gnomad_af")

    A[cols].to_csv(cfg.path("scan_A"), index=False)
    B[cols].to_csv(cfg.path("scan_B"), index=False)
    cp[cols].to_csv(cfg.path("scan_clinvar"), index=False)

    summary = {
        "annotated_with_effect": int(t["efecto"].notna().sum()),
        "with_gnomad_value": int((~t["sin_dato_gnomad"]).sum()),
        "without_gnomad_value": int(t["sin_dato_gnomad"].sum()),
        "gene_assigned_api": int(t["gen_api"].notna().sum()),
        "gene_assigned_local": int(t["gen_local"].notna().sum()),
        "scan_A_rows": int(len(A)),
        "scan_B_genes": int(B["gen"].nunique()),
        "scan_B_rows": int(len(B)),
        "scan_B_genes_with_truncating": genes_trunc,
        "clinvar_pathogenic_coding": int(len(cp)),
    }
    write_json(summary, cfg.path("scan_summary"))
    return summary

In [ ]:
%%writefile mvapipe/stage06_phenotype.py
"""Stage 6: specificity-weighted phenotype ranking, leave-one-symptom-out, permutation null,
and rule-based candidate selection."""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config as cfg
from .utils import write_json


def load_hpo(path=None) -> pd.DataFrame:
    hpo = pd.read_csv(path or cfg.path("hpo"), sep="\t")
    return hpo.drop_duplicates(["hpo_id", "gene_symbol"])


def information_content(hpo: pd.DataFrame, terms) -> tuple[dict, pd.Series, int]:
    """IC(t) = -log2(n_genes(t) / N)."""
    N = hpo["gene_symbol"].nunique()
    n_genes = hpo.groupby("hpo_id")["gene_symbol"].nunique()
    ic = {t: float(-np.log2(n_genes[t] / N)) for t in terms if t in n_genes.index}
    return ic, n_genes, N


def gene_term_matrix(hpo: pd.DataFrame, ic: dict) -> pd.DataFrame:
    return (hpo[hpo["hpo_id"].isin(ic)].assign(v=1)
            .pivot_table(index="gene_symbol", columns="hpo_id", values="v", fill_value=0)
            .reindex(columns=list(ic), fill_value=0))


def damage_base(t: pd.DataFrame) -> pd.DataFrame:
    base = t[t["alto_mod"] & t["rara_o_sin_dato"] & ~t["clinvar_benigno"] & t["gen"].notna()].copy()
    grave = base["efecto"].fillna("").apply(lambda e: bool(set(e.split("&")) & cfg.TRUNCATING))
    base["w"] = np.where(grave | base["clinvar_patog"], cfg.W_STRONG, cfg.W_OTHER)
    return base


def damage_scores(base: pd.DataFrame) -> pd.Series:
    out = {}
    for gen, g in base.groupby("gen"):
        hom = g.loc[g["gt"] == 3, "w"]
        het = g.loc[g["gt"] == 1, "w"].sort_values(ascending=False)
        s_hom = cfg.W_HOM_MULT * hom.max() if len(hom) else 0.0
        s_ch = het.iloc[0] + het.iloc[1] if len(het) >= 2 else 0.0
        s_one = cfg.W_SINGLE_ALLELE * het.iloc[0] if len(het) else 0.0
        out[gen] = max(s_hom, s_ch, s_one)
    return pd.Series(out, dtype=float)


def make_ranker(M: pd.DataFrame, dano: pd.Series):
    def pheno_score(weights: dict) -> pd.Series:
        return (M[list(weights)] * pd.Series(weights)).sum(axis=1)

    def ranking(weights: dict) -> pd.Series:
        ps = pheno_score(weights)
        common = ps.index.intersection(dano.index)
        sc = (ps[common] * dano[common]).sort_values(ascending=False)
        return sc[sc > 0]

    return pheno_score, ranking


def permutation_null(hpo, ic: dict, n_genes: pd.Series, N: int, dano: pd.Series,
                     observed_gene_fit: float, observed_best: float,
                     gene: str, n_perm: int = cfg.PERM_N, seed: int = cfg.PERM_SEED) -> dict:
    """Replace every proband term by a random term of similar specificity and recompute.

    p_gene_fit: probability that `gene` alone reaches its observed phenotype fit.
    p_any_gene_combined: probability that ANY gene reaches the observed best combined score
    (corrects for picking the top gene after the fact)."""
    rng = np.random.default_rng(seed)
    ic_all = -np.log2(n_genes / N)
    term_genes = (hpo[hpo["gene_symbol"].isin(dano.index)]
                  .groupby("hpo_id")["gene_symbol"].apply(list).to_dict())
    twins = {h: ic_all[(ic_all - v).abs() < cfg.IC_TOL].index.difference(list(ic)).tolist()
             for h, v in ic.items()}
    gene_null, best_null = [], []
    for _ in range(n_perm):
        acc: dict[str, float] = {}
        for h in ic:
            if not twins[h]:
                continue
            term = rng.choice(twins[h])
            for g in term_genes.get(term, []):
                acc[g] = acc.get(g, 0.0) + float(ic_all[term])
        best_null.append(max((v * dano[g] for g, v in acc.items()), default=0.0))
        gene_null.append(acc.get(gene, 0.0))
    return {
        "n_permutations": n_perm, "seed": seed, "gene": gene,
        "observed_gene_phenotype_fit": observed_gene_fit,
        "observed_best_combined_score": observed_best,
        "p_gene_fit": float(np.mean(np.array(gene_null) >= observed_gene_fit)),
        "p_any_gene_combined": float(np.mean(np.array(best_null) >= observed_best)),
        "twin_terms_per_symptom": {h: len(v) for h, v in twins.items()},
    }


def select_candidates(t: pd.DataFrame, ranked: pd.Series) -> dict:
    """Primary: top-ranked gene with >= 2 heterozygous candidate alleles (two best form the pair).
    Secondary: truncating ClinVar-pathogenic variants outside that gene, gnomAD missing or rare."""
    base = damage_base(t)
    het = base[base["gt"] == 1]
    primary = None
    for gene in ranked.index:
        g = het[het["gen"] == gene].sort_values(["w", "pos"], ascending=[False, True])
        if len(g) >= 2:
            top = g.head(2).sort_values("pos")
            primary = {"gene": gene,
                       "variants": [{"chrom": str(r.chrom), "pos": int(r.pos), "ref": r.ref,
                                     "alt": r.alt, "efecto": r.efecto, "hgvs_p": r.hgvs_p}
                                    for r in top.itertuples()]}
            break
    secondary = []
    if primary:
        sec = t[t["clinvar_patog"] & ~t["clinvar_benigno"] & (t["gen"] != primary["gene"])
                & t["efecto"].fillna("").apply(lambda e: bool(set(e.split("&")) & cfg.TRUNCATING))
                & (t["gnomad_af"].isna() | (t["gnomad_af"] < cfg.RARE_AF))]
        for r in sec.sort_values("pos").itertuples():
            secondary.append({"gene": r.gen, "chrom": str(r.chrom), "pos": int(r.pos),
                              "ref": r.ref, "alt": r.alt, "efecto": r.efecto, "hgvs_p": r.hgvs_p,
                              "gt": int(r.gt)})
    return {"primary": primary, "secondary": secondary}


def stage06_phenotype(annotated_path=None, hpo_path=None, n_perm: int = cfg.PERM_N) -> dict:
    t = pd.read_parquet(annotated_path or cfg.path("annotated"))
    hpo = load_hpo(hpo_path)
    terms = cfg.load_hpo_terms()
    ic, n_genes, N = information_content(hpo, list(terms))
    M = gene_term_matrix(hpo, ic)
    base = damage_base(t)
    dano = damage_scores(base)
    pheno_score, ranking = make_ranker(M, dano)

    ranked = ranking(ic)
    ranked.rename("combined_score").rename_axis("gene").to_csv(cfg.path("ranking"))

    top_gene = ranked.index[0] if len(ranked) else None
    root = n_genes.get("HP:0000118")

    loo_rows = []
    for h in ic:
        rr = ranking({k: v for k, v in ic.items() if k != h})
        pos = int(list(rr.index).index(top_gene) + 1) if top_gene in rr.index else None
        loo_rows.append({"removed": terms[h], "hpo_id": h,
                         "rank_of_top_gene": pos, "n_scored_genes": len(rr),
                         "top_gene_without_this_symptom": rr.index[0] if len(rr) else None,
                         "score_of_top_gene": float(rr.get(top_gene, np.nan))})
    pd.DataFrame(loo_rows).to_csv(cfg.path("loo"), index=False)

    perm = {}
    if top_gene is not None:
        perm = permutation_null(hpo, ic, n_genes, N, dano,
                                observed_gene_fit=float(pheno_score(ic)[top_gene]),
                                observed_best=float(ranked.iloc[0]),
                                gene=top_gene, n_perm=n_perm)
    write_json(perm, cfg.path("perm"))

    selected = select_candidates(t, ranked)
    write_json(selected, cfg.path("selected"))

    summary = {
        "hpo_genes_N": int(N), "genes_under_root_term": None if root is None else int(root),
        "ic_weights": {terms[h]: round(v, 2) for h, v in ic.items()},
        "genes_scored": int(len(ranked)), "top_gene": top_gene,
        "top_score": float(ranked.iloc[0]) if len(ranked) else None,
        "runner_up": ranked.index[1] if len(ranked) > 1 else None,
        "runner_up_score": float(ranked.iloc[1]) if len(ranked) > 1 else None,
        "top_gene_rank_in_all_leave_one_out": all(r["rank_of_top_gene"] == 1 for r in loo_rows),
        "p_gene_fit": perm.get("p_gene_fit"), "p_any_gene_combined": perm.get("p_any_gene_combined"),
        "primary": selected["primary"], "secondary_n": len(selected["secondary"]),
    }
    write_json(summary, cfg.workdir() / "phenotype_summary.json")
    return summary

In [ ]:
%%writefile mvapipe/stage07_verify.py
"""Stage 7: technical and functional verification of the selected candidates."""
from __future__ import annotations

import re

import pandas as pd
import requests

from . import config as cfg
from .utils import as_list, fmt, hgvs_genomic, maxnum, read_json, write_json

MV_FIELDS = "clinvar,gnomad_genome.af,gnomad_exome.af,dbnsfp,snpeff.ann"


def vcf_call(vcf_path, chrom: str, pos: int, ref: str, alt: str) -> dict:
    """Read one call straight from the VCF (contig names without the 'chr' prefix)."""
    from cyvcf2 import VCF
    chrom = str(chrom).replace("chr", "")
    for v in VCF(str(vcf_path))(f"{chrom}:{pos}-{pos}"):
        if v.POS == pos and v.REF == ref and alt in v.ALT:
            ad = fmt(v, "AD")
            ad_ref, ad_alt = (int(ad[0][0]), int(ad[0][1])) if ad is not None else (None, None)
            tot = (ad_ref + ad_alt) if ad_ref is not None else None
            g = v.genotypes[0]
            return {"found": True, "chrom": chrom, "pos": pos, "ref": v.REF, "alt": list(v.ALT),
                    "filter": v.FILTER or "PASS", "genotype": g[:2], "phased": bool(g[2]),
                    "ad_ref": ad_ref, "ad_alt": ad_alt,
                    "alt_fraction": round(ad_alt / tot, 3) if tot else None,
                    "dp": int(v.gt_depths[0]), "gq": int(v.gt_quals[0]),
                    "has_PGT": fmt(v, "PGT") is not None, "has_PID": fmt(v, "PID") is not None}
    return {"found": False, "chrom": chrom, "pos": pos}


def intervening_hets(qc_df: pd.DataFrame, chrom: str, p1: int, p2: int) -> dict:
    lo, hi = sorted((p1, p2))
    w = qc_df[(qc_df["chrom"] == str(chrom)) & qc_df["pos"].between(lo, hi) & (qc_df["gt"] == 1)] \
        .sort_values("pos")
    gaps = w["pos"].diff().dropna().astype(int).tolist()
    return {"distance_bp": hi - lo, "heterozygous_calls_in_window": int(len(w)),
            "positions": w["pos"].astype(int).tolist(),
            "not_in_dbsnp": w.loc[~w["in_dbsnp"], "pos"].astype(int).tolist(),
            "gaps_bp": gaps, "max_gap_bp": max(gaps) if gaps else None,
            "short_read_phasing_feasible": bool(gaps) and max(gaps) < 300}


def myvariant_detail(chrom, pos, ref, alt, session=None) -> dict:
    http = session or requests
    h = hgvs_genomic(chrom, pos, ref, alt)
    try:
        r = http.get(f"https://myvariant.info/v1/variant/{h}",
                     params={"assembly": "hg38", "fields": MV_FIELDS}, timeout=60)
        if r.status_code != 200:
            return {"hgvs": h, "http": r.status_code}
        j = r.json()
    except Exception as e:
        return {"hgvs": h, "error": str(e)}
    cv = []
    for rcv in as_list((j.get("clinvar") or {}).get("rcv")):
        cond = rcv.get("conditions")
        names = [cond.get("name")] if isinstance(cond, dict) else [c.get("name") for c in as_list(cond)]
        cv.append({"significance": rcv.get("clinical_significance"), "conditions": names,
                   "review_status": rcv.get("review_status"),
                   "submitters": rcv.get("number_submitters")})
    db = j.get("dbnsfp") or {}
    cadd = db.get("cadd") if isinstance(db.get("cadd"), dict) else {}
    ann = as_list((j.get("snpeff") or {}).get("ann"))
    return {
        "hgvs": h, "clinvar": cv,
        "gnomad_genome_af": maxnum(((j.get("gnomad_genome") or {}).get("af") or {}).get("af")),
        "gnomad_exome_af": maxnum(((j.get("gnomad_exome") or {}).get("af") or {}).get("af")),
        "predictors": {
            "cadd_phred": maxnum(cadd.get("phred")), "revel": maxnum(db.get("revel")),
            "alphamissense": maxnum(db.get("alphamissense")),
            "polyphen2_hdiv": maxnum((db.get("polyphen2") or {}).get("hdiv")),
            "sift": maxnum(db.get("sift")), "metarnn": maxnum(db.get("metarnn")),
            "phylop_100way": maxnum((db.get("phylop") or {}).get("100way_vertebrate")),
            "phastcons_100way": maxnum((db.get("phastcons") or {}).get("100way_vertebrate"))},
        "uniprot_acc": [u.get("acc") for u in as_list(db.get("uniprot")) if isinstance(u, dict)],
        "snpeff_effects": [a.get("effect") for a in ann][:6],
    }


def uniprot_features(accession: str, residue: int, window: int = 15, session=None) -> dict:
    http = session or requests
    try:
        r = http.get(f"https://rest.uniprot.org/uniprotkb/{accession}.json", timeout=60)
        if r.status_code != 200:
            return {"accession": accession, "http": r.status_code}
        j = r.json()
    except Exception as e:
        return {"accession": accession, "error": str(e)}
    feats = []
    for ft in j.get("features", []):
        s, e = ft["location"]["start"].get("value"), ft["location"]["end"].get("value")
        if s is None or e is None:
            continue
        if ft["type"] in ("Domain", "Region", "Motif", "Active site", "Binding site",
                          "Natural variant", "Mutagenesis", "Modified residue") \
                and (s <= residue <= e or abs(s - residue) <= window or abs(e - residue) <= window):
            feats.append({"type": ft["type"], "start": s, "end": e,
                          "description": (ft.get("description") or "")[:140],
                          "contains_residue": s <= residue <= e})
    return {"accession": accession, "length": j["sequence"]["length"], "residue": residue,
            "features_near_residue": feats}


def residue_from_hgvs_p(h: str | None) -> int | None:
    m = re.search(r"p\.[A-Za-z]{3}(\d+)", h or "")
    return int(m.group(1)) if m else None


def stage07_verify(session=None) -> dict:
    sel = read_json(cfg.path("selected"))
    qc_df = pd.read_parquet(cfg.path("qc"))
    out: dict = {"primary": None, "secondary": []}
    prim = sel.get("primary")
    if prim:
        calls = [vcf_call(cfg.path("vcf"), v["chrom"], v["pos"], v["ref"], v["alt"])
                 for v in prim["variants"]]
        detail = [myvariant_detail(v["chrom"], v["pos"], v["ref"], v["alt"], session)
                  for v in prim["variants"]]
        v1, v2 = prim["variants"][0], prim["variants"][1]
        phase = intervening_hets(qc_df, v1["chrom"], v1["pos"], v2["pos"])
        phase["any_phase_tags"] = any(c.get("phased") or c.get("has_PGT") or c.get("has_PID")
                                      for c in calls if c.get("found"))
        uni = []
        for v, d in zip(prim["variants"], detail):
            res = residue_from_hgvs_p(v.get("hgvs_p"))
            acc = (d.get("uniprot_acc") or [None])[0]
            if res and acc:
                uni.append(uniprot_features(acc.split("-")[0], res, session=session))
        out["primary"] = {"gene": prim["gene"], "variants": prim["variants"], "vcf_calls": calls,
                          "annotation": detail, "pair_geometry_and_phase": phase, "uniprot": uni}
    for s in sel.get("secondary", []):
        out["secondary"].append({
            "variant": s,
            "vcf_call": vcf_call(cfg.path("vcf"), s["chrom"], s["pos"], s["ref"], s["alt"]),
            "annotation": myvariant_detail(s["chrom"], s["pos"], s["ref"], s["alt"], session)})
    write_json(out, cfg.path("verification"))
    return out

In [ ]:
%%writefile mvapipe/stage08_locus.py
"""Stage 8 (supplementary): unfiltered review of the primary candidate's whole locus.

All records in the gene body plus flanks are read from the VCF with NO quality filter and each is
classified by position (CDS, UTR exon, intron, flank). This asks whether anything outside coding
sequence, or anything our quality filters removed, could complete or contradict the coding pair.
"""
from __future__ import annotations

from collections import Counter

import numpy as np
import pandas as pd
import requests

from . import config as cfg
from .utils import (flat_nums, fmt, gtf_iter, hgvs_genomic, in_intervals, maxnum, merge_intervals,
                    read_json, write_json)


def gene_structure(gtf_path, gene: str) -> dict | None:
    span = None
    exons, cds = [], []
    for chrom, feat, s, e, strand, attrs in gtf_iter(gtf_path, {"gene", "exon", "CDS"}):
        if f'gene_name "{gene}"' not in attrs:
            continue
        if feat == "gene":
            span = (chrom, s, e, strand)
        elif feat == "exon":
            exons.append((s, e))
        else:
            cds.append((s, e))
    if span is None:
        return None
    return {"chrom": span[0], "start": span[1], "end": span[2], "strand": span[3],
            "exons": merge_intervals(exons), "cds": merge_intervals(cds)}


def classify(pos: int, g: dict) -> tuple[str, int | None]:
    p = np.array([pos])
    if in_intervals(*g["cds"], p)[0]:
        cls = "CDS"
    elif in_intervals(*g["exons"], p)[0]:
        cls = "UTR_exon"
    elif g["start"] <= pos <= g["end"]:
        cls = "intron"
    else:
        up = pos < g["start"] if g["strand"] == "+" else pos > g["end"]
        cls = "upstream_flank" if up else "downstream_flank"
    s, e = g["exons"]
    edges = np.concatenate([s, e]) if len(s) else np.array([])
    dist = int(np.min(np.abs(edges - pos))) if len(edges) else None
    return cls, dist


def read_locus(vcf_path, chrom: str, start: int, end: int) -> pd.DataFrame:
    from cyvcf2 import VCF
    rows = []
    for v in VCF(str(vcf_path))(f"{chrom}:{start}-{end}"):
        ad = fmt(v, "AD")
        rows.append({"chrom": str(v.CHROM), "pos": v.POS, "ref": v.REF, "alt": ",".join(v.ALT),
                     "qual": v.QUAL, "filter": v.FILTER or "PASS", "gt": int(v.gt_types[0]),
                     "dp": int(v.gt_depths[0]), "gq": int(v.gt_quals[0]),
                     "ad_ref": int(ad[0][0]) if ad is not None else -1,
                     "ad_alt": int(ad[0][1]) if ad is not None else -1,
                     "in_dbsnp": bool(v.INFO.get("DB", False))})
    return pd.DataFrame(rows)


def annotate_rare(df: pd.DataFrame, cap: int = 300, session=None) -> pd.DataFrame:
    """Add gnomAD AF and CADD for non-dbSNP records (best effort; failures are recorded)."""
    http = session or requests
    df = df.copy()
    df["gnomad_af"] = np.nan
    df["cadd"] = np.nan
    df["annotation_status"] = "not_requested"
    sub = df[(~df["in_dbsnp"]) & (df["gt"].isin([1, 3]))].head(cap)
    ids = {i: hgvs_genomic(r.chrom, r.pos, r.ref, r.alt) for i, r in sub.iterrows()}
    ids = {i: h for i, h in ids.items() if h}
    if not ids:
        return df
    try:
        r = http.post("https://myvariant.info/v1/variant",
                      data={"ids": ",".join(ids.values()), "assembly": "hg38",
                            "fields": "gnomad_genome.af.af,gnomad_exome.af.af,dbnsfp.cadd.phred,cadd.phred"},
                      timeout=120)
        r.raise_for_status()
        by_q = {x.get("query"): x for x in r.json()}
    except Exception as e:
        df.loc[list(ids), "annotation_status"] = f"failed: {type(e).__name__}"
        return df
    for i, h in ids.items():
        x = by_q.get(h, {})
        if x.get("notfound"):
            df.at[i, "annotation_status"] = "not_found"
            continue
        afs = []
        for k in ("gnomad_genome", "gnomad_exome"):
            afs += flat_nums(((x.get(k) or {}).get("af") or {}).get("af"))
        df.at[i, "gnomad_af"] = max(afs) if afs else np.nan
        cadd = (x.get("dbnsfp") or {}).get("cadd")
        df.at[i, "cadd"] = maxnum(cadd.get("phred")) if isinstance(cadd, dict) \
            else maxnum((x.get("cadd") or {}).get("phred"))
        df.at[i, "annotation_status"] = "annotated"
    return df


def stage08_locus(gene: str | None = None, vcf_path=None, gtf_path=None,
                  flank: int = cfg.LOCUS_FLANK, session=None, annotate: bool = True) -> dict:
    gene = gene or (read_json(cfg.path("selected")).get("primary") or {}).get("gene")
    if not gene:
        raise RuntimeError("No primary gene selected (run stage 6 first).")
    g = gene_structure(gtf_path or cfg.path("gtf"), gene)
    if g is None:
        raise RuntimeError(f"Gene {gene} not found in the GTF")
    df = read_locus(vcf_path or cfg.path("vcf"), g["chrom"], max(1, g["start"] - flank), g["end"] + flank)
    cls = [classify(int(p), g) for p in df["pos"]]
    df["region"] = [c[0] for c in cls]
    df["dist_to_exon_edge"] = [c[1] for c in cls]
    df["splice_region_pm8"] = (df["region"] == "intron") & (df["dist_to_exon_edge"].fillna(10**9) <= 8)
    if annotate:
        df = annotate_rare(df, session=session)
    else:
        df["gnomad_af"], df["cadd"], df["annotation_status"] = np.nan, np.nan, "not_requested"
    df.to_csv(cfg.path("locus"), index=False)

    nonref = df[df["gt"].isin([1, 3])]
    brief = ["pos", "ref", "alt", "gt", "filter", "dp", "gq", "in_dbsnp"]
    summary = {
        "gene": gene, "chrom": g["chrom"], "gene_start": g["start"], "gene_end": g["end"],
        "strand": g["strand"], "flank_bp": flank,
        "records_total": int(len(df)), "records_non_reference": int(len(nonref)),
        "by_region_non_reference": dict(Counter(nonref["region"])),
        "by_filter_non_reference": dict(Counter(nonref["filter"])),
        "non_reference_not_in_dbsnp": int((~nonref["in_dbsnp"]).sum()),
        "coding_non_reference": nonref[nonref["region"] == "CDS"][brief].to_dict("records"),
        "splice_region_non_reference": nonref[nonref["splice_region_pm8"]][brief].to_dict("records"),
        "non_pass_non_reference": nonref[nonref["filter"] != "PASS"][brief].to_dict("records"),
        "rare_noncoding_candidates": nonref[(nonref["region"] != "CDS") & (~nonref["in_dbsnp"])
                                            & (nonref["gnomad_af"].isna() | (nonref["gnomad_af"] < cfg.RARE_AF))]
        [["pos", "ref", "alt", "gt", "region", "dist_to_exon_edge", "dp", "gq", "gnomad_af", "cadd",
          "annotation_status"]].to_dict("records"),
    }
    write_json(summary, cfg.path("locus_summary"))
    return summary

In [ ]:
%%writefile mvapipe/stage09_10_nmd_gnomad.py
"""Stages 9-10 (supplementary): nonsense-mediated decay position check and gnomAD v4 lookup."""
from __future__ import annotations

import requests

from . import config as cfg
from .utils import gtf_iter, parse_attrs, read_json, write_json

NMD_RULE_NT = 50


def pick_transcript(gtf_path, gene: str) -> dict | None:
    """MANE Select; else Ensembl canonical; else the transcript with most exon bases."""
    tx: dict[str, dict] = {}
    for chrom, feat, s, e, strand, attrs in gtf_iter(gtf_path, {"transcript", "exon"}):
        if f'gene_name "{gene}"' not in attrs:
            continue
        a = parse_attrs(attrs)
        tid = a.get("transcript_id")
        d = tx.setdefault(tid, {"id": tid, "chrom": chrom, "strand": strand, "exons": [], "tags": []})
        if feat == "transcript":
            tags = a.get("tag", [])
            d["tags"] = tags if isinstance(tags, list) else [tags]
        else:
            d["exons"].append((s, e))
    if not tx:
        return None
    for want in ("MANE_Select", "Ensembl_canonical"):
        hit = [d for d in tx.values() if want in d["tags"]]
        if hit:
            hit[0]["selected_by"] = want
            return hit[0]
    best = max(tx.values(), key=lambda d: sum(e - s + 1 for s, e in d["exons"]))
    best["selected_by"] = "longest_exon_total"
    return best


def nmd_position(tx: dict, pos: int) -> dict:
    """Exon index and distance from `pos` to the last exon-exon junction, in spliced transcript space."""
    ex = sorted(tx["exons"], key=lambda x: x[0], reverse=(tx["strand"] == "-"))
    n = len(ex)
    offset, running, hit_idx = None, 0, None
    for i, (s, e) in enumerate(ex):
        if s <= pos <= e:
            offset = running + ((pos - s) if tx["strand"] == "+" else (e - pos))
            hit_idx = i + 1
        running += e - s + 1
    if offset is None:
        return {"in_exon": False}
    last_len = ex[-1][1] - ex[-1][0] + 1
    last_junction = running - last_len
    dist = last_junction - offset
    in_last = hit_idx == n
    return {"in_exon": True, "exon_number": hit_idx, "total_exons": n,
            "transcript_offset_nt": offset, "mrna_length_nt": running,
            "distance_to_last_junction_nt": dist, "in_last_exon": in_last,
            "predicted_nmd": bool((not in_last) and dist > NMD_RULE_NT),
            "interpretation": ("last exon: expected to escape NMD" if in_last else
                               f"{dist} nt upstream of the last junction: "
                               + ("NMD expected (>50 nt rule)" if dist > NMD_RULE_NT
                                  else "within 50 nt of the last junction: may escape NMD"))}


def stage09_nmd(gtf_path=None) -> dict:
    sel = read_json(cfg.path("selected"))
    gtf_path = gtf_path or cfg.path("gtf")
    results, targets = [], []
    if sel.get("primary"):
        targets += [dict(v, gene=sel["primary"]["gene"]) for v in sel["primary"]["variants"]]
    targets += sel.get("secondary", [])
    for v in targets:
        if not set(str(v.get("efecto", "")).split("&")) & cfg.TRUNCATING:
            continue
        tx = pick_transcript(gtf_path, v["gene"])
        if tx is None:
            results.append({"variant": v, "error": "gene not found"})
            continue
        r = nmd_position(tx, int(v["pos"]))
        r.update({"variant": v, "transcript": tx["id"], "transcript_selected_by": tx["selected_by"],
                  "caveat": "approximate: uses the variant position as the premature stop position "
                            "(error of at most 2 nt); frameshift stops lie further downstream"})
        results.append(r)
    write_json({"nmd_rule_nt": NMD_RULE_NT, "results": results}, cfg.path("nmd"))
    return {"n_checked": len(results)}


GNOMAD_URL = "https://gnomad.broadinstitute.org/api"
GNOMAD_QUERY = """
query($id: String!, $ds: DatasetId!) {
  variant(variantId: $id, dataset: $ds) {
    variant_id
    exome { ac an homozygote_count }
    genome { ac an homozygote_count }
  }
}"""


def gnomad_lookup(chrom: str, pos: int, ref: str, alt: str, dataset: str = "gnomad_r4", session=None) -> dict:
    """Query the gnomAD GraphQL API. Never raises: returns a status instead."""
    http = session or requests
    vid = f"{str(chrom).replace('chr', '')}-{pos}-{ref}-{alt}"
    try:
        r = http.post(GNOMAD_URL, json={"query": GNOMAD_QUERY, "variables": {"id": vid, "ds": dataset}},
                      timeout=60)
    except Exception as e:
        return {"variant_id": vid, "status": "unavailable", "detail": f"{type(e).__name__}: {e}"}
    if r.status_code != 200:
        return {"variant_id": vid, "status": "unavailable", "detail": f"HTTP {r.status_code}"}
    j = r.json()
    v = (j.get("data") or {}).get("variant")
    if v is None:
        msgs = [e.get("message", "") for e in j.get("errors", [])]
        if any("not found" in m.lower() for m in msgs):
            return {"variant_id": vid, "status": "absent_from_gnomad_v4", "detail": msgs}
        return {"variant_id": vid, "status": "unavailable", "detail": msgs or "empty response"}
    out = {"variant_id": vid, "status": "present"}
    for k in ("exome", "genome"):
        d = v.get(k)
        out[k] = None if not d else {"ac": d["ac"], "an": d["an"],
                                     "af": (d["ac"] / d["an"]) if d["an"] else None,
                                     "homozygotes": d.get("homozygote_count")}
    return out


def stage10_gnomad(session=None) -> dict:
    sel = read_json(cfg.path("selected"))
    variants = []
    if sel.get("primary"):
        variants += sel["primary"]["variants"]
    variants += sel.get("secondary", [])
    res = [gnomad_lookup(v["chrom"], v["pos"], v["ref"], v["alt"], session=session) for v in variants]
    write_json({"dataset": "gnomad_r4", "results": res,
                "note": "'unavailable' means the query could not be completed, NOT that the variant is absent. "
                        "Check unavailable entries manually in the gnomAD browser."}, cfg.path("gnomad"))
    return {"n": len(res), "statuses": [r["status"] for r in res]}

In [ ]:
%%writefile mvapipe/stage11_baf.py
"""Stage 11 (supplementary): allele-fraction (B-allele frequency) and depth screen for chromosome imbalance.

Heterozygous-site allele fractions are a second, partly independent signal besides depth: in cells with an
extra or missing chromosome they drift away from 0.5. Both are summarised per chromosome and per 5 Mb
window as robust z-scores against the genome-wide distribution, and the smallest UNIFORM mosaic fraction
the depth signal could detect is estimated.

LIMITATION: in mosaic variegated aneuploidy the affected chromosomes differ from cell to cell, so a
single chromosome is imbalanced in few cells. A null result bounds what a bulk test could see; it does
not exclude the condition.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config as cfg
from .utils import write_json

AUTOSOMES = [str(i) for i in range(1, 23)]


def robust_z(x: pd.Series) -> pd.Series:
    med = x.median()
    mad = (x - med).abs().median() * 1.4826
    return (x - med) / mad if mad and mad > 0 else x * 0.0


def trimmed_mean(s: pd.Series, lo: float = 0.05, hi: float = 0.95) -> float:
    """Mean between the lo and hi quantiles; unlike the median of integer depths it is not quantised."""
    a, b = s.quantile([lo, hi])
    return float(s[(s >= a) & (s <= b)].mean())


def het_snv_table(qc: pd.DataFrame) -> pd.DataFrame:
    d = qc[(qc["gt"] == 1) & (qc["ref"].str.len() == 1) & (qc["alt"].str.len() == 1)
           & (~qc["alt"].str.contains(",")) & (qc["dp"] >= cfg.BAF_MIN_DP)].copy()
    tot = d["ad_ref"] + d["ad_alt"]
    d = d[(d["ad_ref"] >= 0) & (tot > 0)]
    d["alt_frac"] = d["ad_alt"] / (d["ad_ref"] + d["ad_alt"])
    d = d[d["dp"] <= 2 * d["dp"].median()]
    d["dev"] = (d["alt_frac"] - 0.5).abs()
    return d


def stage11_baf(qc_path=None, bin_size: int = cfg.BIN_SIZE) -> dict:
    qc = pd.read_parquet(qc_path or cfg.path("qc"))
    het = het_snv_table(qc)
    chroms = AUTOSOMES + ["X"]

    g = het[het["chrom"].isin(chroms)].groupby("chrom")
    chrom_tab = pd.DataFrame({
        "n_het_snv": g.size(),
        "dp_trimmed_mean": g["dp"].apply(trimmed_mean),
        "median_alt_frac": g["alt_frac"].median(),
        "mean_abs_dev_from_0.5": g["dev"].mean(),
        "frac_outside_0.4_0.6": g["alt_frac"].apply(lambda s: float(((s < 0.4) | (s > 0.6)).mean())),
    }).reindex(chroms)
    auto = chrom_tab.loc[AUTOSOMES]
    chrom_tab["depth_ratio"] = chrom_tab["dp_trimmed_mean"] / auto["dp_trimmed_mean"].median()
    chrom_tab["z_depth"] = robust_z(chrom_tab.loc[AUTOSOMES, "depth_ratio"]).reindex(chroms)
    chrom_tab["z_baf_dev"] = robust_z(chrom_tab.loc[AUTOSOMES, "mean_abs_dev_from_0.5"]).reindex(chroms)
    chrom_tab["flag"] = (chrom_tab["z_depth"].abs() > 3.5) | (chrom_tab["z_baf_dev"].abs() > 3.5)
    chrom_tab.to_csv(cfg.path("baf_chrom"))

    h = het[het["chrom"].isin(AUTOSOMES)].copy()
    h["bin"] = (h["pos"] // bin_size).astype(int)
    gb = h.groupby(["chrom", "bin"])
    bins = pd.DataFrame({"n_het_snv": gb.size(), "dp_trimmed_mean": gb["dp"].apply(trimmed_mean),
                         "mean_abs_dev": gb["dev"].mean()}).reset_index()
    bins = bins[bins["n_het_snv"] >= cfg.BAF_MIN_SITES_PER_BIN].copy()
    bins["depth_ratio"] = bins["dp_trimmed_mean"] / bins["dp_trimmed_mean"].median()
    bins["z_depth"] = robust_z(bins["depth_ratio"])
    bins["z_baf_dev"] = robust_z(bins["mean_abs_dev"])
    bins["start"] = bins["bin"] * bin_size
    bins["flag"] = (bins["z_depth"].abs() > 3.5) | (bins["z_baf_dev"].abs() > 3.5)
    bins.to_csv(cfg.path("baf_bins"), index=False)

    # uniform mosaic fraction f shifts depth ratio by ~ f/2 (trisomy +f/2, monosomy -f/2)
    ratios = auto["dp_trimmed_mean"] / auto["dp_trimmed_mean"].median()
    spread = max(float(ratios.std()), float((ratios - 1).abs().median() * 1.4826))
    summary = {
        "het_snv_used": int(len(het)), "bin_size_bp": bin_size, "bins_tested": int(len(bins)),
        "chromosomes_flagged": chrom_tab.index[chrom_tab["flag"].fillna(False)].tolist(),
        "bins_flagged": int(bins["flag"].sum()),
        "flagged_bins_detail": bins[bins["flag"]][["chrom", "start", "n_het_snv", "depth_ratio", "z_depth",
                                                   "z_baf_dev"]].round(3).to_dict("records")[:20],
        "autosome_depth_ratio_spread": round(spread, 4),
        "approx_min_detectable_uniform_mosaic_fraction_depth": round(min(1.0, 2 * 3.5 * spread), 4),
        "x_depth_ratio": round(float(chrom_tab.loc["X", "depth_ratio"]), 3),
        "x_frac_het_among_all_calls": None,
        "caveat": "Bulk sample; MVA aneuploidies vary cell to cell, so each chromosome is imbalanced in few "
                  "cells. Only variants passing QC contribute. The detection limit is approximate and "
                  "assumes a uniform mosaic fraction.",
    }
    xs = qc[qc["chrom"] == "X"]
    if len(xs):
        summary["x_frac_het_among_all_calls"] = round(float((xs["gt"] == 1).mean()), 3)
    write_json(summary, cfg.path("baf_summary"))
    return summary

In [ ]:
%%writefile mvapipe/stage12_submission.py
"""Stage 12: write the Track 1 submission CSV and validate it.

Schema (from the challenge Space): one row per prediction; a compound-heterozygous PAIR is ONE row
(chrom_1..alt_1 and chrom_2..alt_2). Splitting a pair into two rows earns only partial credit.
Coordinates are GRCh38 with the 'chr' prefix; `epcr` in (0, 1] orders rows (highest first).
"""
from __future__ import annotations

import csv
import re

from . import config as cfg
from .utils import read_json

COLUMNS = ["proband_id", "chrom_1", "pos_1", "ref_1", "alt_1", "chrom_2", "pos_2", "ref_2", "alt_2",
           "epcr", "finding_type", "notes"]


def _chr(c: str) -> str:
    c = str(c)
    return c if c.startswith("chr") else "chr" + c


def build_rows(sel: dict) -> list[dict]:
    rows = []
    p = sel.get("primary")
    if p:
        v1, v2 = p["variants"][0], p["variants"][1]
        rows.append({"proband_id": cfg.PROBAND_ID,
                     "chrom_1": _chr(v1["chrom"]), "pos_1": v1["pos"], "ref_1": v1["ref"], "alt_1": v1["alt"],
                     "chrom_2": _chr(v2["chrom"]), "pos_2": v2["pos"], "ref_2": v2["ref"], "alt_2": v2["alt"],
                     "epcr": cfg.EPCR_PRIMARY, "finding_type": "primary",
                     "notes": f"{p['gene']} compound-heterozygous candidate: {v1.get('hgvs_p')} + "
                              f"{v2.get('hgvs_p')}. Phase unresolved."})
    for i, s in enumerate(sel.get("secondary", [])[: cfg.MAX_ROWS - len(rows)]):
        rows.append({"proband_id": cfg.PROBAND_ID,
                     "chrom_1": _chr(s["chrom"]), "pos_1": s["pos"], "ref_1": s["ref"], "alt_1": s["alt"],
                     "chrom_2": "", "pos_2": "", "ref_2": "", "alt_2": "",
                     "epcr": round(max(cfg.EPCR_SECONDARY - 0.01 * i, 0.01), 2), "finding_type": "secondary",
                     "notes": f"{s['gene']} {s.get('hgvs_p')} heterozygous; possible incidental finding."})
    return rows


def write_csv(rows: list[dict], path) -> None:
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLUMNS)
        w.writeheader()
        w.writerows(rows)


def validate_submission(path, vcf_path=None) -> list[str]:
    """Return a list of problems (empty = valid)."""
    problems: list[str] = []
    with open(path, newline="") as f:
        rd = csv.DictReader(f)
        missing = [c for c in COLUMNS[:10] if c not in (rd.fieldnames or [])]
        if missing:
            return [f"missing columns: {missing}"]
        rows = list(rd)
    if not rows:
        problems.append("no rows")
    if len(rows) > cfg.MAX_ROWS:
        problems.append(f"more than {cfg.MAX_ROWS} rows")
    prev, seen = 2.0, set()
    for i, r in enumerate(rows, 1):
        tag = f"row {i}"
        if r["proband_id"].strip() != cfg.PROBAND_ID:
            problems.append(f"{tag}: proband_id must be {cfg.PROBAND_ID}")
        try:
            e = float(r["epcr"])
            if not 0 < e <= 1:
                problems.append(f"{tag}: epcr must be in (0, 1]")
            if e > prev:
                problems.append(f"{tag}: rows are not ordered by decreasing epcr")
            prev = e
        except ValueError:
            problems.append(f"{tag}: epcr is not a number")
        if r.get("finding_type", "") not in ("", "primary", "secondary"):
            problems.append(f"{tag}: finding_type must be primary or secondary")
        variants = []
        for k in ("1", "2"):
            c, p, ref, alt = (r[f"chrom_{k}"].strip(), r[f"pos_{k}"].strip(),
                              r[f"ref_{k}"].strip().upper(), r[f"alt_{k}"].strip().upper())
            if not any((c, p, ref, alt)):
                continue
            if not all((c, p, ref, alt)):
                problems.append(f"{tag}: variant {k} is incomplete")
                continue
            if not re.fullmatch(r"chr([1-9]|1[0-9]|2[0-2]|X|Y|M)", c):
                problems.append(f"{tag}: chrom_{k} '{c}' must look like chr15 (GRCh38, 'chr' prefix)")
            if not p.isdigit():
                problems.append(f"{tag}: pos_{k} is not an integer")
            if not re.fullmatch(r"[ACGT]+", ref) or not re.fullmatch(r"[ACGT]+", alt):
                problems.append(f"{tag}: ref_{k}/alt_{k} must contain only A, C, G, T")
            variants.append((c, p, ref, alt))
        if not variants:
            problems.append(f"{tag}: no variant")
        key = frozenset(variants)
        if key in seen:
            problems.append(f"{tag}: duplicate prediction")
        seen.add(key)
        if vcf_path:
            from cyvcf2 import VCF
            for c, p, ref, alt in variants:
                if not p.isdigit():
                    continue
                found = any(v.POS == int(p) and v.REF == ref and alt in v.ALT
                            for v in VCF(str(vcf_path))(f"{c.replace('chr', '')}:{p}-{p}"))
                if not found:
                    problems.append(f"{tag}: {c}:{p} {ref}>{alt} is not in the VCF")
    return problems


def stage12_submission(out=None, verify_against_vcf: bool = True) -> dict:
    sel = read_json(cfg.path("selected"))
    rows = build_rows(sel)
    out = out or cfg.path("submission")
    write_csv(rows, out)
    problems = validate_submission(out, cfg.path("vcf") if verify_against_vcf else None)
    return {"rows": len(rows), "path": str(out), "problems": problems, "valid": not problems}

In [ ]:
%%writefile expected_results.json
{
 "_comment": "Numbers reported in the methods report (interactive session). check_results.py compares a fresh run with them; [value, tolerance].",
 "qc_stats.json": {
  "kept": [
   4669249,
   0
  ],
  "het": [
   2889730,
   0
  ],
  "hom_alt": [
   1779519,
   0
  ]
 },
 "coding_stats.json": {
  "coding_variants": [
   26252,
   0
  ],
  "coding_het": [
   16748,
   0
  ],
  "coding_hom_alt": [
   9504,
   0
  ]
 },
 "annotation_stats.json": {
  "multiallelic_not_annotated": [
   50,
   0
  ],
  "notfound": [
   6,
   3
  ]
 },
 "scan_summary.json": {
  "gene_assigned_local": [
   26103,
   5
  ],
  "scan_A_rows": [
   2,
   0
  ],
  "scan_B_genes": [
   8,
   0
  ],
  "scan_B_rows": [
   20,
   2
  ],
  "clinvar_pathogenic_coding": [
   9,
   1
  ]
 },
 "phenotype_summary.json": {
  "hpo_genes_N": [
   5276,
   50
  ],
  "genes_scored": [
   24,
   3
  ],
  "top_score": [
   20.88,
   0.1
  ],
  "runner_up_score": [
   8.01,
   0.1
  ],
  "p_gene_fit": [
   0.008,
   0.01
  ],
  "p_any_gene_combined": [
   0.0265,
   0.015
  ]
 }
}

In [ ]:
%%writefile check_results.py
#!/usr/bin/env python3
"""Compare a fresh pipeline run with the numbers reported in the methods report.

    python check_results.py [--workdir DIR]

A difference is not necessarily an error (live services and the 'latest' HPO release can change),
but each one must be explained before the report is submitted.
"""
import argparse, json, os, sys
from pathlib import Path


def main() -> int:
    ap = argparse.ArgumentParser(); ap.add_argument("--workdir", default=None)
    args = ap.parse_args()
    if args.workdir:
        os.environ["MVA_WORKDIR"] = args.workdir
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from mvapipe import config as cfg
    with open(Path(__file__).with_name("expected_results.json")) as fh:
        exp = json.load(fh)
    wd, bad = cfg.workdir(), 0

    def load(name):
        p = wd / name
        if not p.exists():
            return None
        with open(p) as fh:
            return json.load(fh)

    print(f"{'item':58} {'expected':>12} {'observed':>12}  status")
    for fname, items in exp.items():
        if fname.startswith("_") or fname in ("text", "sets"):
            continue
        data = load(fname)
        for key, (val, tol) in items.items():
            obs = None if data is None else data.get(key)
            ok = obs is not None and abs(obs - val) <= tol
            bad += not ok
            print(f"{fname + ':' + key:58} {val:>12} {str(obs)[:12]:>12}  {'PASS' if ok else 'DIFF'}")
    for fname, items in exp.get("text", {}).items():
        data = load(fname) or {}
        for key, val in items.items():
            ok = data.get(key) == val
            bad += not ok
            print(f"{fname + ':' + key:58} {val:>12} {str(data.get(key)):>12}  {'PASS' if ok else 'DIFF'}")
    for fname, items in exp.get("sets", {}).items():
        data = load(fname) or {}
        for key, val in items.items():
            ok = sorted(data.get(key, [])) == sorted(val)
            bad += not ok
            print(f"{fname + ':' + key:58} {'set':>12} {'match' if ok else 'differs':>12}  {'PASS' if ok else 'DIFF'}")
    print("\nAll items match." if not bad else f"\n{bad} item(s) differ: explain each before submitting.")
    return 1 if bad else 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
import json
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
from mvapipe import config as cfg
from mvapipe.utils import hf_token, read_json
import mvapipe.stage01_03_data_qc_coding as s1
import mvapipe.stage04_05_annotate_scan as s4
import mvapipe.stage06_phenotype as s6
import mvapipe.stage07_verify as s7
import mvapipe.stage08_locus as s8
import mvapipe.stage09_10_nmd_gnomad as s9
import mvapipe.stage11_baf as s11
import mvapipe.stage12_submission as s12
assert hf_token(), "Add HF_TOKEN as a Kaggle secret (and accept the dataset terms) before continuing"
print("working directory:", cfg.workdir())

## 1. Data and provenance
VCF, index and phenotype document from Hugging Face; Ensembl GTF (GRCh38, r112); HPO gene–phenotype table. Records sizes and SHA-256 and checks the HPO terms against the document.

In [ ]:
prov = s1.stage01_data()
print("HPO terms match the document:", prov.get("hpo_terms_match_config"))
pd.DataFrame(prov["files"]).T

## 2. Quality control
PASS, non-reference, DP ≥ 10, GQ ≥ 30. **Every exclusion is counted** (the interactive run recorded only the final count).

In [ ]:
pd.Series(s1.stage02_qc())

## 3. Coding restriction
CDS ±2 bp (canonical splice sites), merged across isoforms.

In [ ]:
pd.Series(s1.stage03_coding())

## 4. Annotation
Batched MyVariant.info queries (hg38). Multi-allelic records are not annotated and are counted. Raw JSON is cached.

In [ ]:
pd.Series(s4.stage04_annotate())

## 5. Phenotype-agnostic recessive scan (pre-specified filters)
Impact high/moderate; gnomAD AF < 0.1 % **or no value** (kept, flagged); no benign ClinVar. Genes assigned locally from the GTF. A = homozygous; B = ≥2 heterozygous in one gene. Check X-linked 'homozygous' calls (hemizygous in XY individuals) and genes whose pair is an indel cluster (possible artefact).

In [ ]:
scan = s4.stage05_scan()
print(json.dumps(scan, indent=2))
print("\n--- A: homozygous ---"); display(pd.read_csv(cfg.path("scan_A")))
print("--- B: >=2 heterozygous in one gene ---"); display(pd.read_csv(cfg.path("scan_B")))

## 6. Phenotype ranking, robustness and permutation null
HPO terms weighted by IC = −log₂(genes with term / all genes). Gene score = phenotype fit × damage score. Leave-one-symptom-out and a permutation null (random terms of similar specificity): (i) chance of the top gene's fit, (ii) chance that **any** gene reaches the best score (corrects for post-hoc selection). Selection is rule-based (no gene names). *Caveat:* damage weights are not blind.

In [ ]:
ph = s6.stage06_phenotype()
print(json.dumps({k: v for k, v in ph.items() if k != "primary"}, indent=2, default=str))
print("\nTop of ranking:"); display(pd.read_csv(cfg.path("ranking")).head(10))
print("Leave-one-symptom-out:"); display(pd.read_csv(cfg.path("loo")))
print("Selected:"); print(json.dumps(read_json(cfg.path("selected")), indent=2))

## 7. Verification of the candidates
Call read from the VCF (allele balance, depth, GQ, FILTER, phasing tags); heterozygous markers between the pair members and the largest gap (can short reads phase them?); ClinVar, predictors, UniProt features.

In [ ]:
ver = s7.stage07_verify()
p = ver["primary"]
if p:
    print("Primary gene:", p["gene"])
    display(pd.DataFrame(p["vcf_calls"])[["chrom","pos","ref","alt","filter","genotype","phased","ad_ref","ad_alt","alt_fraction","dp","gq","has_PGT","has_PID"]])
    print(json.dumps(p["pair_geometry_and_phase"], indent=2))
    for a in p["annotation"]:
        print("\n", a.get("hgvs"), "| gnomAD genome/exome AF:", a.get("gnomad_genome_af"), a.get("gnomad_exome_af"))
        print("  ClinVar:", json.dumps(a.get("clinvar"))[:600]); print("  predictors:", a.get("predictors"))
    for u in p["uniprot"]:
        print("\nUniProt", u.get("accession"), "residue", u.get("residue")); display(pd.DataFrame(u.get("features_near_residue", [])))

## 8. Supplementary — unfiltered locus review
Every record in the gene body ±50 kb with **no quality filter**, classified (CDS, UTR exon, intron, flank). Is there a non-coding/splice-region second hit? Did a quality filter remove a coding call?

In [ ]:
loc = s8.stage08_locus()
print(json.dumps({k: v for k, v in loc.items() if k not in ("coding_non_reference","rare_noncoding_candidates")}, indent=2, default=str))
print("\nCoding records, unfiltered:"); display(pd.DataFrame(loc["coding_non_reference"]))
print("Rare non-coding candidates:"); display(pd.DataFrame(loc["rare_noncoding_candidates"]))

## 9. Supplementary — exon position and NMD rule
MANE Select transcript. A premature stop >~50 nt upstream of the last exon–exon junction is expected to trigger NMD; a stop in the last exon is expected to escape it.

In [ ]:
s9.stage09_nmd()
for r in read_json(cfg.path("nmd"))["results"]:
    print(r["variant"].get("gene"), r["variant"].get("hgvs_p"), "->", r.get("transcript"), "| exon", r.get("exon_number"), "of", r.get("total_exons"), "|", r.get("interpretation"))

## 10. Supplementary — gnomAD v4 lookup
**`unavailable` means the query failed, not that the variant is absent.** Check such entries in the gnomAD browser and note the result by hand.

In [ ]:
print(s9.stage10_gnomad())
display(pd.json_normalize(read_json(cfg.path("gnomad"))["results"]))

## 11. Supplementary — allele-fraction and depth screen
Het-SNV allele fractions and depth per chromosome and 5 Mb window (robust z-scores) and the smallest *uniform* mosaic fraction detectable in depth. In MVA the aneuploidies differ cell to cell: a null result bounds a bulk test, it does **not** exclude the condition.

In [ ]:
bs = s11.stage11_baf()
print(json.dumps(bs, indent=2, default=str))
display(pd.read_csv(cfg.path("baf_chrom"), index_col=0).round(4))

## 12. Submission file
One row per prediction; the compound-heterozygous pair is **one row**; GRCh38 with `chr` prefix. The validator checks the schema and that each variant exists in the VCF.

In [ ]:
sub = s12.stage12_submission()
print(sub)
display(pd.read_csv(cfg.path("submission")))

## 13. Compare with the reported numbers

In [ ]:
!python check_results.py

## 14. Files
Download `submission.csv`. **Do not make variant-level files public** (`variants_*.parquet`, `myvariant_raw.json`, `locus_review.csv`, `annotated.parquet`).

In [ ]:
for f in sorted(cfg.workdir().glob("*.json")) + sorted(cfg.workdir().glob("*.csv")):
    print(f"{f.stat().st_size:>10,d}  {f.name}")